# Explicación Detallada de Modelos y Modularización en FatigueSet

Este notebook sirve como guía metodológica y técnica para entender cómo se han diseñado, modularizado y persistido los modelos de aprendizaje automático en el proyecto **FatigueSet**.

---

## 1. Estructura y Filosofía de Modularización

Siguiendo las mejores prácticas de ingeniería de software en Machine Learning (ML), hemos estructurado el código del paquete de forma modular en `fatigueset-lib/`:

*   **`fatigueset.loader`**: Carga de datos fisiológicos brutos.
*   **`fatigueset.processor`**: Sincronización y normalización de series temporales.
*   **`fatigueset.models`**: Carpeta o subpaquete que contiene las definiciones de los modelos. Específicamente, el archivo `rnn.py` ha sido reubicado aquí (`fatigueset/models/rnn.py`).

Además, todos los pesos entrenados y serializaciones se guardan de forma centralizada en una carpeta raíz llamada `/models/`.

In [1]:
# Verificando la carga de los módulos reubicados en fatigueset.models
import sys
from pathlib import Path

# Añadir la librería local fatigueset-lib al path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset.models.rnn import FatigueSequenceDataset, RNNFatiga, train_kfold
print("✓ Subpaquete 'fatigueset.models' cargado e importado con éxito.")

✓ Subpaquete 'fatigueset.models' cargado e importado con éxito.


## 2. Preparación de Datos Secuenciales: `FatigueSequenceDataset`

La clase `FatigueSequenceDataset` hereda de `torch.utils.data.Dataset`. Se encarga de respaldar las secuencias temporales en forma de tensores listos para la Red Neuronal Recurrente (RNN).

### Mecanismo de Ventanas y Alineación Fisiológica:
1.  **Alineación temporal (`_merge_raw_streams`):** Se alinean las señales del pecho (Zephyr) y de la muñeca (Empatica) usando `pd.merge_asof` por marca de tiempo más cercana, ordenando por participante y sesión para evitar mezclar registros.
2.  **Construcción de secuencias (`_build_sequences`):** Se extraen ventanas deslizantes de tamaño `seq_len` y desplazamiento `step` dentro del historial temporal de cada sesión. Esto genera tensores de dimensiones `(muestras, seq_len, features)`. 
3.  **Evitar Fugas de Datos (Data Leakage):** Para prevenir que información de un participante afecte a las predicciones de otro, se mantiene la variable `groups` (que identifica al participante de cada secuencia), la cual es consumida posteriormente por `GroupKFold` para realizar validaciones cruzadas estrictas.

In [2]:
# Visualización del Dataset de PyTorch
import numpy as np

# Generamos datos sintéticos simulados para ver el funcionamiento de la clase
X_mock = np.random.randn(10, 8, 15)  # 10 muestras, 8 pasos temporales, 15 características
y_mock = np.random.randn(10, 2)     # Targets: fatiga física y fatiga mental

dataset = FatigueSequenceDataset(X_mock, y_mock)
print(f"Tamaño del Dataset: {len(dataset)} muestras")
x_sample, y_sample = dataset[0]
print(f"Dimensión de una secuencia (X): {x_sample.shape}")
print(f"Dimensión del target (y): {y_sample.shape}")

Tamaño del Dataset: 10 muestras
Dimensión de una secuencia (X): (8, 15)
Dimensión del target (y): (2,)


## 3. Arquitectura del Modelo: `RNNFatiga`

El modelo `RNNFatiga` hereda de `nn.Module`. Consiste en:
1.  **Capa RNN clásica (`nn.RNN`):** Recibe las características de entrada y las procesa de forma secuencial, acumulando la información en un estado oculto.
2.  **Capa Totalmente Conectada (`nn.Linear`):** Toma la salida del último paso temporal de la secuencia y la proyecta a las 2 variables continuas objetivo (fatiga física y mental).

### Código y Flujo de la Red:
```python
class RNNFatiga(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, 2)  # Predice física y mental
        
    def forward(self, x):
        out, _ = self.rnn(x)
        # Retornamos la predicción a partir de la salida del último paso temporal: out[:, -1, :]
        return self.fc(out[:, -1, :])
```

In [3]:
import torch

# Inicializamos un modelo simulado
input_features = 15
model = RNNFatiga(input_size=input_features, hidden_size=32, num_layers=1)
print("=== Estructura de la RNN ===")
print(model)

# Probamos una pasada hacia adelante (Forward Pass)
batch_tensor = torch.randn(4, 8, input_features)  # Batch de 4 secuencias
predictions = model(batch_tensor)
print(f"\nDimensión de salida de la predicción: {predictions.shape} -> [Batch, (Física, Mental)]")

=== Estructura de la RNN ===
RNNFatiga(
  (rnn): RNN(15, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=2, bias=True)
)

Dimensión de salida de la predicción: torch.Size([4, 2]) -> [Batch, (Física, Mental)]


## 4. Bucle de Entrenamiento y Validación Cruzada: `train_kfold`

El entrenamiento se realiza a través de la función `train_kfold` en `rnn.py`. Sus componentes claves son:

### A. Validación Cruzada por Participante (`GroupKFold`)
Para garantizar la capacidad de generalización en sujetos no vistos durante el entrenamiento, se realiza una validación cruzada basada en grupos, donde cada participante pertenece exclusivamente al conjunto de entrenamiento o validación/prueba en cada Fold.

### B. El bucle de entrenamiento estándar de PyTorch
Para cada época, se itera sobre los lotes del DataLoader:
1.  **Forward pass:** Predicción sobre la ventana de entrada.
2.  **Cálculo de Pérdida (`nn.MSELoss`):** Comparación con la fatiga real.
3.  **optimizer.zero_grad():** Reseteo de gradientes acumulados.
4.  **loss.backward():** Propagación hacia atrás para calcular gradientes.
5.  **Grad clipping:** `torch.nn.utils.clip_grad_norm_` para evitar gradientes explosivos.
6.  **optimizer.step():** Actualización de pesos usando Adam.

### C. Early Stopping y Checkpoints
Si la pérdida de validación no mejora durante 10 épocas consecutivas (`patience=10`), se detiene el entrenamiento de ese Fold de forma anticipada. Los pesos del mejor modelo se guardan en la ruta `/models/rnn/model_fold_{fold}.pt`.

## 5. Serialización de Modelos Clásicos (Scikit-Learn)

En los modelos clásicos (Random Forest, SVM, KNN, Regresores Lineales/Regulados) entrenados mediante `run_models_classicos.py`, se ha incorporado la serialización usando la librería estándar `pickle`.

Esto permite persistir los estimadores ya entrenados en archivos `.pkl` dentro del directorio `/models/classicos/`, facilitando su posterior carga y uso en predicción o despliegue:

```python
import pickle
import re

# Sanitizar el nombre del modelo para usarlo como nombre de archivo seguro
nombre_sanitizado = re.sub(r'[^a-zA-Z0-9_]', '_', nombre_modelo.lower())
ruta_archivo = f"models/classicos/{nombre_sanitizado}.pkl"

# Guardar el modelo en disco
with open(ruta_archivo, 'wb') as f:
    pickle.dump(modelo_entrenado, f)
```

In [4]:
import pickle
from sklearn.ensemble import RandomForestRegressor

# Ejemplo conceptual de creación y guardado de un modelo clásico
bosque_juguete = RandomForestRegressor(n_estimators=10, random_state=42)
# Simulamos datos
X_toy = np.random.randn(20, 5)
y_toy = np.random.randn(20)
bosque_juguete.fit(X_toy, y_toy)

# Serializar a bytes
modelo_serializado = pickle.dumps(bosque_juguete)
print(f"Modelo clásico serializado. Tamaño en bytes: {len(modelo_serializado)}")

# Reconstruir el modelo desde la serialización
bosque_recargado = pickle.loads(modelo_serializado)
pred_toy = bosque_recargado.predict(X_toy[:2])
print("Predicciones del modelo recargado:", pred_toy)

Modelo clásico serializado. Tamaño en bytes: 20762
Predicciones del modelo recargado: [-0.16161811 -0.68999778]
